In [2]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from models import BayesianChangePointDetector

# Load data
analysis_df = pd.read_csv(r'C:\Users\hp\Pictures\brent-change-point\brent-change-point\data\processed_oil_prices.csv', 
                         index_col='date', parse_dates=True)

# Use log returns for stationarity
data = analysis_df['log_returns'].dropna().values
dates = analysis_df['log_returns'].dropna().index

# Initialize and fit model
detector = BayesianChangePointDetector(n_samples=3000, n_tune=1000)
detector.fit(data, n_change_points=1)

# Diagnose model
summary = detector.diagnose()

# Plot traces
fig_trace = detector.plot_trace()
plt.show()

# Get change points
change_points = detector.get_change_points(dates)
print(f"Most probable change point: {change_points['mode_date']}")
print(f"Probability: {change_points['probability']:.2%}")

# Plot posterior
fig_posterior = detector.plot_posterior(dates, data)
plt.show()

# Associate with events
events = pd.read_csv(r'c:\Users\hp\Pictures\brent-change-point\brent-change-point\data\processed_events.csv', parse_dates=['date'])
for _, event in events.iterrows():
    days_diff = abs((event['date'] - change_points['mode_date']).days)
    if days_diff <= 30:  # Within 30 days
        print(f"Event near change point ({days_diff} days): {event['Event_Description']}")

KeyError: 'log_returns'

In [4]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from models import BayesianChangePointDetector

# Load data
analysis_df = pd.read_csv(
    r'C:\Users\hp\Pictures\brent-change-point\brent-change-point\data\processed_oil_prices.csv',
    index_col='date',
    parse_dates=True
)

# Ensure correct price column name
price_col = 'price' if 'price' in analysis_df.columns else 'Price'

# Create log returns
analysis_df['log_price'] = np.log(analysis_df[price_col])
analysis_df['log_returns'] = analysis_df['log_price'].diff()
analysis_df = analysis_df.dropna()

data = analysis_df['log_returns'].values
dates = analysis_df.index

# Initialize and fit model
detector = BayesianChangePointDetector(n_samples=3000, n_tune=1000)
detector.fit(data)

# Diagnose model
summary = detector.diagnose()

# Plot traces
detector.plot_trace("traceplot.png")

# Get change point results
results = detector.get_change_point_results(dates)

print(f"Most probable change point: {results['mode_date']}")
print(f"Posterior probability: {results['posterior_probability']:.2%}")

# Plot posterior
detector.plot_posterior(dates, data, filename="posterior_plot.png")


Multiprocess sampling (2 chains in 4 jobs)
CompoundStep
>Metropolis: [tau]
>NUTS: [mu1, mu2, sigma]


Sampling 2 chains for 1_000 tune and 3_000 draw iterations (2_000 + 6_000 draws total) took 128 seconds.
There were 181 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Model Summary:
        mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  ess_tail  \
tau    0.718  0.450   0.000    1.000      0.020    0.014     522.0     522.0   
mu1    0.073  0.042  -0.023    0.147      0.001    0.001    1606.0    1192.0   
mu2    0.033  0.029  -0.019    0.092      0.001    0.001    1302.0    2495.0   
sigma  0.029  0.018   0.003    0.061      0.001    0.001     156.0      34.0   

       r_hat  
tau     1.00  
mu1     1.00  
mu2     1.00  
sigma   1.02  

R-hat values (should be < 1.1):
       r_hat
tau     1.00
mu1     1.00
mu2     1.00
sigma   1.02
Trace plot saved to traceplot.png
Most probable change point: 2021-10-01 00:00:00
Posterior probability: 71.82%
Posterior plot saved to posterior_plot.png


<Figure size 1200x800 with 4 Axes>